## Libraries & Organizing

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_BATCHES =  PROJECT_ROOT / "data" / "batches"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

## Datasets

### CARS

In [ ]:
car_immediate = pd.read_csv(DATA_RAW / "car_60_5_1.csv")
car_medium = pd.read_csv(DATA_RAW / "car_130_10_5.csv")
car_long = pd.read_csv(DATA_RAW / "car_260_25_10.csv")

### Scores

In [ ]:
model = 'llama' #enter 'llama' or 'qwen' here.

scores = pd.read_csv(DATA_PROCESSED / f"output_{model}_parsed.csv") #enter filename here

In [ ]:
car_immediate.rename(columns={'evtdate': 'date', 'car': 'car_immediate'}, inplace=True)
car_medium.rename(columns={'evtdate': 'date', 'car': 'car_medium'}, inplace=True)
car_long.rename(columns={'evtdate': 'date', 'car': 'car_long'}, inplace=True)

In [ ]:
car_immediate.columns
car_medium.columns
car_long.columns
scores.columns

In [ ]:
scores = scores.merge(
    car_immediate[['permno','date', 'car_immediate']], on=["permno", "date"], how="left").merge(
    car_medium[['permno', 'date', 'car_medium']], on=["permno", "date"], how="left").merge(
        car_long[['permno', 'date', 'car_long']], on=["permno", "date"], how="left")

scores.drop(columns=['tokens'], inplace=True)

In [ ]:
scores.isna().sum()

In [ ]:
dict =pd.read_csv(DATA_PROCESSED / "dictionary_output.csv", index_col=0)
print(dict.columns)
dict.rename(columns={'transcriptid': 'transcript_id', 'fli_score': 'dict_score'}, inplace=True)

In [ ]:
scores.dropna(subset=['permno'], inplace=True)

In [ ]:
scores = scores.merge(dict[['transcript_id', 'dict_score']], on='transcript_id', how='left')

In [ ]:
controls = pd.read_csv(DATA_PROCESSED / "controls_extensive.csv")

In [ ]:
controls.columns


In [ ]:
controls.isna().sum()

In [ ]:
final = scores.merge(controls[['permno', 'date','bm', 'log_assets','gind', 'word_count', 'log_word_count']], on=["permno", "date"], how="left")


In [ ]:
final.isna().sum()

In [ ]:
final = final.drop_duplicates(subset=["transcript_id"])

In [ ]:
final.describe()

In [ ]:
final["date"] = pd.to_datetime(final["date"])

final["year"] = final["date"].dt.year

In [ ]:
if model == 'llama':
    final.to_csv(DATA_PROCESSED / "final_dataset_NAs.csv", index=False)
else:    final.to_csv(DATA_PROCESSED / f"final_dataset_{model}.csv", index=False)